# 临床机器学习示范

本教程演示如何使用机器学习技术进行临床预测分析。以**糖尿病风险预测**为应用场景。

## 1. 环境准备

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('环境准备完成!')

## 2. 数据生成

生成1000名患者的模拟临床数据集。

In [ ]:
def generate_clinical_data(n_samples=1000, random_state=42):
    np.random.seed(random_state)
    age = np.random.normal(55, 12, n_samples).clip(18, 90).astype(int)
    gender = np.random.choice(['男', '女'], n_samples)
    bmi = np.random.normal(25, 4, n_samples).clip(16, 45)
    glucose = 4 + 0.1 * age + 0.15 * bmi + np.random.normal(0, 1, n_samples)
    glucose = glucose.clip(3, 20)
    hba1c = 3 + 0.4 * glucose + np.random.normal(0, 0.5, n_samples)
    hba1c = hba1c.clip(3, 15)
    systolic_bp = 100 + 0.3 * age + 0.5 * bmi + np.random.normal(0, 10, n_samples)
    systolic_bp = systolic_bp.clip(90, 200).astype(int)
    diastolic_bp = 60 + 0.1 * age + 0.3 * bmi + np.random.normal(0, 8, n_samples)
    diastolic_bp = diastolic_bp.clip(50, 120).astype(int)
    cholesterol = 150 + 0.5 * age + 2 * bmi + np.random.normal(0, 30, n_samples)
    cholesterol = cholesterol.clip(100, 350)
    triglycerides = 100 + 0.3 * age + 3 * bmi + np.random.normal(0, 40, n_samples)
    triglycerides = triglycerides.clip(50, 500)
    hdl = 50 - 0.1 * bmi + np.random.normal(0, 10, n_samples)
    hdl = hdl.clip(20, 100)
    ldl = cholesterol * 0.6 + np.random.normal(0, 15, n_samples)
    ldl = ldl.clip(50, 250)
    creatinine = 0.8 + 0.005 * age + np.random.normal(0, 0.2, n_samples)
    creatinine = creatinine.clip(0.4, 3.0)
    diabetes_prob = (
        0.02 * (age - 40) + 0.05 * (bmi - 25) + 0.15 * (glucose - 6) +
        0.1 * (hba1c - 5.5) + 0.01 * (systolic_bp - 120) +
        np.random.normal(0, 0.3, n_samples)
    )
    diabetes_prob = 1 / (1 + np.exp(-diabetes_prob))
    diabetes = (diabetes_prob > 0.5).astype(int)
    data = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, n_samples + 1)],
        'age': age, 'gender': gender,
        'bmi': bmi.round(1), 'glucose': glucose.round(1),
        'hba1c': hba1c.round(1),
        'systolic_bp': systolic_bp, 'diastolic_bp': diastolic_bp,
        'cholesterol': cholesterol.round(1),
        'triglycerides': triglycerides.round(1),
        'hdl': hdl.round(1), 'ldl': ldl.round(1),
        'creatinine': creatinine.round(2), 'diabetes': diabetes
    })
    return data

df = generate_clinical_data(n_samples=1000)
print(f'数据集大小: {df.shape}')
print(f'糖尿病患者比例: {df["diabetes"].mean()*100:.1f}%')
df.head()

In [ ]:
df.to_csv('clinical_data.csv', index=False, encoding='utf-8-sig')
print('数据已保存到 clinical_data.csv')

## 3. 数据探索性分析 (EDA)

In [ ]:
print('数据集基本信息:')
print('=' * 50)
print(f'样本数量: {len(df)}')
print(f'特征数量: {len(df.columns) - 2}')
print(f'数据类型:')
print(df.dtypes)

In [ ]:
print('数值特征统计摘要:')
df.describe().round(2)

In [ ]:
# 目标变量分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
diabetes_counts = df['diabetes'].value_counts()
axes[0].pie(diabetes_counts, labels=['正常', '糖尿病'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('糖尿病患者比例')
axes[1].bar(['正常', '糖尿病'], diabetes_counts, color=['#2ecc71', '#e74c3c'])
axes[1].set_xlabel('类别')
axes[1].set_ylabel('人数')
axes[1].set_title('糖尿病患者数量')
for i, v in enumerate(diabetes_counts):
    axes[1].text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# 特征分布
numeric_features = ['age', 'bmi', 'glucose', 'hba1c', 'systolic_bp', 'cholesterol']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, feature in enumerate(numeric_features):
    axes[i].hist(df[feature], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('频数')
    axes[i].set_title(f'{feature} 分布')
    axes[i].axvline(df[feature].mean(), color='red', linestyle='--',
                    label=f'均值: {df[feature].mean():.1f}')
    axes[i].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 特征 vs 糖尿病 (箱线图)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, feature in enumerate(numeric_features):
    df.boxplot(column=feature, by='diabetes', ax=axes[i])
    axes[i].set_xlabel('糖尿病状态')
    axes[i].set_ylabel(feature)
    axes[i].set_title(f'{feature} vs 糖尿病')
    axes[i].set_xticklabels(['正常', '糖尿病'])
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# 相关性热力图
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', linewidths=0.5)
plt.title('特征相关性热力图')
plt.tight_layout()
plt.show()

## 4. 数据预处理

In [ ]:
feature_columns = ['age', 'bmi', 'glucose', 'hba1c', 'systolic_bp', 'diastolic_bp',
                   'cholesterol', 'triglycerides', 'hdl', 'ldl', 'creatinine']
X = df[feature_columns].copy()
y = df['diabetes'].copy()
print(f'特征矩阵形状: {X.shape}')
print(f'目标变量形状: {y.shape}')

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'训练集大小: {X_train.shape[0]}')
print(f'测试集大小: {X_test.shape[0]}')
print(f'训练集糖尿病比例: {y_train.mean()*100:.1f}%')
print(f'测试集糖尿病比例: {y_test.mean()*100:.1f}%')

In [ ]:
# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns)
print('标准化后的训练集统计:')
X_train_scaled.describe().round(2)

## 5. 模型训练

训练两种模型：逻辑回归（可解释性强）和随机森林（性能好）。

In [ ]:
print('训练逻辑回归模型...')
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
print('逻辑回归模型训练完成!')

In [ ]:
print('训练随机森林模型...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
print('随机森林模型训练完成!')

## 6. 模型评估

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    print(f'{model_name} 评估结果:')
    print('=' * 50)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    print(f'准确率 (Accuracy):  {accuracy:.4f}')
    print(f'精确率 (Precision): {precision:.4f}')
    print(f'召回率 (Recall):    {recall:.4f}')
    print(f'F1分数 (F1-Score):  {f1:.4f}')
    print(f'AUC-ROC:            {auc:.4f}')
    return {'accuracy': accuracy, 'precision': precision,
            'recall': recall, 'f1': f1, 'auc': auc}

lr_metrics = evaluate_model(y_test, y_pred_lr, y_prob_lr, '逻辑回归')
rf_metrics = evaluate_model(y_test, y_pred_rf, y_prob_rf, '随机森林')

In [ ]:
# 分类报告
print('逻辑回归分类报告:')
print(classification_report(y_test, y_pred_lr, target_names=['正常', '糖尿病']))
print('随机森林分类报告:')
print(classification_report(y_test, y_pred_rf, target_names=['正常', '糖尿病']))

In [ ]:
# 混淆矩阵
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['正常', '糖尿病'], yticklabels=['正常', '糖尿病'])
axes[0].set_xlabel('预测值')
axes[0].set_ylabel('真实值')
axes[0].set_title('逻辑回归混淆矩阵')
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['正常', '糖尿病'], yticklabels=['正常', '糖尿病'])
axes[1].set_xlabel('预测值')
axes[1].set_ylabel('真实值')
axes[1].set_title('随机森林混淆矩阵')
plt.tight_layout()
plt.show()

In [ ]:
# ROC曲线
fig, ax = plt.subplots(figsize=(8, 6))
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
ax.plot(fpr_lr, tpr_lr, 'b-', linewidth=2,
        label=f'逻辑回归 (AUC = {lr_metrics["auc"]:.3f})')
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
ax.plot(fpr_rf, tpr_rf, 'r-', linewidth=2,
        label=f'随机森林 (AUC = {rf_metrics["auc"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='随机猜测')
ax.set_xlabel('假阳性率 (FPR)')
ax.set_ylabel('真阳性率 (TPR)')
ax.set_title('ROC曲线')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

In [ ]:
# 交叉验证
print('交叉验证结果 (5折):')
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f'逻辑回归 AUC: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std() * 2:.4f})')
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f'随机森林 AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std() * 2:.4f})')

## 7. 特征重要性分析

In [ ]:
# 逻辑回归系数
lr_coef = pd.DataFrame({
    'feature': feature_columns,
    'coefficient': lr_model.coef_[0]
})
lr_coef['abs_coef'] = abs(lr_coef['coefficient'])
lr_coef = lr_coef.sort_values('abs_coef', ascending=True)
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in lr_coef['coefficient']]
plt.barh(lr_coef['feature'], lr_coef['coefficient'], color=colors)
plt.xlabel('系数值')
plt.ylabel('特征')
plt.title('逻辑回归特征系数')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 随机森林特征重要性
rf_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)
plt.figure(figsize=(10, 6))
plt.barh(rf_importance['feature'], rf_importance['importance'], color='steelblue')
plt.xlabel('重要性')
plt.ylabel('特征')
plt.title('随机森林特征重要性')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 模型应用 - 预测新患者

In [ ]:
def predict_diabetes_risk(patient_data, model, scaler, feature_columns):
    patient_df = pd.DataFrame([patient_data])
    patient_features = patient_df[feature_columns]
    patient_scaled = scaler.transform(patient_features)
    prediction = model.predict(patient_scaled)[0]
    probability = model.predict_proba(patient_scaled)[0][1]
    return prediction, probability

new_patient = {'age': 55, 'bmi': 28.5, 'glucose': 7.2, 'hba1c': 6.8,
               'systolic_bp': 145, 'diastolic_bp': 90, 'cholesterol': 220,
               'triglycerides': 180, 'hdl': 45, 'ldl': 130, 'creatinine': 1.0}

pred_lr, prob_lr = predict_diabetes_risk(new_patient, lr_model, scaler, feature_columns)
pred_rf, prob_rf = predict_diabetes_risk(new_patient, rf_model, scaler, feature_columns)
print('预测结果:')
print(f'逻辑回归: {"糖尿病" if pred_lr == 1 else "正常"} (风险概率: {prob_lr*100:.1f}%)')
print(f'随机森林: {"糖尿病" if pred_rf == 1 else "正常"} (风险概率: {prob_rf*100:.1f}%)')

In [ ]:
# 批量预测
test_patients = [
    {'age': 35, 'bmi': 22.0, 'glucose': 5.5, 'hba1c': 5.2, 'systolic_bp': 115,
     'diastolic_bp': 75, 'cholesterol': 180, 'triglycerides': 100, 'hdl': 55, 'ldl': 100, 'creatinine': 0.8},
    {'age': 65, 'bmi': 32.0, 'glucose': 8.5, 'hba1c': 7.5, 'systolic_bp': 155,
     'diastolic_bp': 95, 'cholesterol': 250, 'triglycerides': 220, 'hdl': 38, 'ldl': 160, 'creatinine': 1.2},
    {'age': 50, 'bmi': 26.0, 'glucose': 6.8, 'hba1c': 6.2, 'systolic_bp': 130,
     'diastolic_bp': 82, 'cholesterol': 200, 'triglycerides': 150, 'hdl': 48, 'ldl': 120, 'creatinine': 0.9},
]
print('批量预测结果:')
print(f'{"年龄":<6} {"BMI":<8} {"血糖":<8} {"预测":<8} {"风险概率":<10}')
for i, pt in enumerate(test_patients):
    pred, prob = predict_diabetes_risk(pt, rf_model, scaler, feature_columns)
    status = '糖尿病' if pred == 1 else '正常'
    print(f'{pt["age"]:<6} {pt["bmi"]:<8} {pt["glucose"]:<8} {status:<8} {prob*100:.1f}%')

## 9. 模型性能对比

In [ ]:
metrics_names = ['准确率', '精确率', '召回率', 'F1分数', 'AUC']
lr_values = [lr_metrics['accuracy'], lr_metrics['precision'], lr_metrics['recall'],
             lr_metrics['f1'], lr_metrics['auc']]
rf_values = [rf_metrics['accuracy'], rf_metrics['precision'], rf_metrics['recall'],
             rf_metrics['f1'], rf_metrics['auc']]
x = np.arange(len(metrics_names))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, lr_values, width, label='逻辑回归', color='#3498db')
bars2 = ax.bar(x + width/2, rf_values, width, label='随机森林', color='#e74c3c')
ax.set_xlabel('评估指标')
ax.set_ylabel('分数')
ax.set_title('模型性能对比')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim([0.5, 1.0])
ax.grid(axis='y', alpha=0.3)
for bars in [bars1, bars2]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 10. 临床应用建议

### 模型选择
- **逻辑回归**: 适合需要高可解释性的临床场景
- **随机森林**: 通常具有更好的预测性能

### 重要预测因子
1. 空腹血糖 (glucose) - 最重要的预测因子
2. 糖化血红蛋白 (hba1c) - 反映长期血糖控制
3. BMI - 肥胖是糖尿病的重要风险因素
4. 年龄 - 年龄增长增加患病风险

### 注意事项
1. 本模型使用模拟数据，实际应用需使用真实临床数据重新训练
2. 模型预测结果仅供参考，不能替代临床医生的专业判断
3. 在临床应用前需要进行充分的外部验证

---
**返回**: [Python基础培训](01_python_basics.ipynb)